In [1]:
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()


In [2]:
import requests
import pandas as pd
import certifi
import yfinance as yf
import math
import ta


url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; ResearchNotebook/1.0)"
}

response = requests.get(
    url,
    headers=headers,
    verify=certifi.where(),
    timeout=10
)
response.raise_for_status()

tables = pd.read_html(response.text)
sp500 = tables[0]

TICKERS = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()

len(TICKERS), TICKERS[:10]
START_DATE = "2018-01-01"

/var/folders/wz/vg2971s50r937v89lnqmjxdh0000gn/T/ipykernel_72814/1443508198.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [4]:
def compute_score(df):
    close = df["Close"].squeeze()  # ensures 1D Series
    ma40 = close.rolling(40).mean()
    rsi = ta.momentum.RSIIndicator(close, window=14).rsi()
    
    score = (
        (close > ma40).astype(int) * 2 +
        (rsi < 70).astype(int) * 1 +
        ((close / close.shift(12)) - 1) * 10
    )
    
    return score.iloc[-1]


In [5]:
def batch_download(tickers, start_date=START_DATE):
    batch_size = 50
    all_data = {}
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        print(f"Downloading batch {i//batch_size+1}/{math.ceil(len(tickers)/batch_size)}...")
        data = yf.download(batch, start=start_date, interval="1d", auto_adjust=True, progress=False)
        
        # If only one ticker, data["Close"] is Series, wrap in DataFrame
        if len(batch) == 1:
            df_close = data["Close"].to_frame(name="Close")
            all_data[batch[0]] = df_close
        else:
            # Multiple tickers: MultiIndex, extract each ticker
            for t in batch:
                df_close = data["Close"][t].dropna().to_frame(name="Close")
                all_data[t] = df_close
    
    return all_data


In [6]:
all_data = batch_download(TICKERS)

candidates = []

for ticker, df in all_data.items():
    if len(df) < 52:  # at least 1 year of weekly data
        continue
    try:
        score = compute_score(df)
        candidates.append({
            "Ticker": ticker,
            "Score": score,
            "Price": df["Close"].iloc[-1]
        })
    except Exception as e:
        print(f"Skipping {ticker}: {e}")
        continue

# Rank top 10
top10 = pd.DataFrame(candidates).sort_values("Score", ascending=False).head(10)
top10


,Ticker,Score,Price
395,HOOD,5.759204,105.709999
251,INTC,5.504658,140.940002
317,MRNA,5.096412,59.345001
417,LUV,4.935918,48.570000
39,AMAT,4.783913,640.179993
213,GEV,4.759570,1127.589966
277,KLAC,4.665697,269.160004
385,RL,4.432547,410.920013
403,SNDK,4.414578,2273.729980
84,CAH,4.372989,222.740005


In [7]:
top10 = (
    pd.DataFrame(candidates)
    .sort_values("Score", ascending=False)
    .head(10)
)

top10


,Ticker,Score,Price
395,HOOD,5.759204,105.709999
251,INTC,5.504658,140.940002
317,MRNA,5.096412,59.345001
417,LUV,4.935918,48.570000
39,AMAT,4.783913,640.179993
213,GEV,4.759570,1127.589966
277,KLAC,4.665697,269.160004
385,RL,4.432547,410.920013
403,SNDK,4.414578,2273.729980
84,CAH,4.372989,222.740005
